# ETF期待リターン統合アンサンブル（Rolling IC Softmax）

本ノートは、複数手法で得られるETF期待リターンを動的統合し、`EQW`・expert単体・統合戦略を同一条件で比較する。

- 参照ノートは変更しない（非破壊）。
- 因子特徴量は常に `t` までで計算し、`t+1` へ適用する。
- 統合対象は一意予測のみ（重複予測は除外） + `VOLMIX`。


## 1) 仮説と数式

### 仮説
- 異なる推定器の予測を動的重み付けすれば、単一手法より安定した性能が得られる。
- 手法間の不一致が大きい銘柄は、期待リターンを縮小した方が頑健である。

### 定義
expert $e$ の予測を $
\hat r_{i,t}^{(e)}$ とし、同時点横断で標準化して
\[
\tilde r_{i,t}^{(e)}=\mathrm{RankZ}(\hat r_{i,t}^{(e)})
\]
とする。

直近窓 $W$ で expert スキルを
\[
IC_{e,\tau}=\mathrm{corr}_{cs}(\tilde r_{\cdot,\tau}^{(e)}, r_{\cdot,\tau+1})
\]
\[
s_{e,t}=\overline{IC}_{e,t}^{(W)}-\lambda_{to}\overline{TO}_{e,t}^{(W)}-\lambda_{\sigma}\sigma(IC_{e,t}^{(W)})
\]
で定義。

softmax 重みは
\[
\omega_{e,t}=\frac{\exp(\tau s_{e,t})}{\sum_k \exp(\tau s_{k,t})}
\]
。

統合予測は
\[
\hat r_{i,t}^{ens}=\sum_e \omega_{e,t}\tilde r_{i,t}^{(e)},\quad
 d_{i,t}=\mathrm{std}_e(\tilde r_{i,t}^{(e)})
\]
\[
\hat r_{i,t}^{final}=\frac{\hat r_{i,t}^{ens}}{1+\gamma d_{i,t}}
\]
。

配分は `proportional` と `meanvariance_tc` を比較する。


In [ ]:
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, Lasso
from scipy.optimize import nnls, minimize
from IPython.display import display

np.random.seed(42)

CONFIG = {
    # Data scope
    'start_date': '2010-07-23',
    'end_date': None,
    'tickers': [
        'AGNG','AIQ','AQWA','BITS','BKCH','BOTZ','BUG','CHPX','CLOU','CTEC','DRIV','DTCR',
        'EBIZ','FINX','GNOM','HEAL','HERO','HYDR','IPAV','KROP','LIT','MILN','PAVE','RNRG',
        'SHLD','SNSR','SOCL','ZAP'
    ],
    'local_csv_path': 'data/theme_etf.csv',
    'rebalance': 'W-FRI',

    # Backtest
    'cost_bps': 10,
    'benchmark': 'EQW',

    # PCA / factor construction
    'pca_window_months': 60,
    'n_components': 8,
    'min_obs_per_window': 24,
    'factor_mom_lookbacks': [1, 12],
    'factor_mom_skip_months': 1,
    'combine_weights': {1: 0.5, 12: 0.5},
    'top_m_factors': 3,
    'exclude_pcs': [1],

    # Exposure estimation
    'regression_window_months': 60,
    'exposure_standardize_x': True,
    'ridge_alpha': 5.0,
    'lasso_alpha': 0.001,
    'lasso_max_iter': 10000,
    'nnls_fit_intercept': True,

    # Allocation constraints
    'long_only': True,
    'weight_cap': None,
    'etf_top_k': 5,

    # Mean-Variance-TC
    'vol_window_months': 6,
    'cov_window_months': 12,
    'cov_shrinkage': 0.2,
    'mv_risk_aversion': 10.0,
    'mv_turnover_penalty': 5.0,
    'mv_abs_eps': 1e-8,
    'mv_solver_maxiter': 300,

    # Theme heat (sign-aware expert生成で使用)
    'heat_dispersion_weight': 0.3333333333,
    'heat_breadth_weight': 0.3333333333,
    'heat_volume_weight': 0.3333333333,
    'heat_signal_alpha': 0.50,
    'heat_alloc_beta': 0.50,
    'heat_signal_floor': 0.20,
    'heat_local_blend': 0.50,
    'heat_norm_lookback_months': 24,
    'heat_norm_min_periods_months': 12,
    'heat_volume_z_window_months': 6,

    # Volume conditioned short momentum
    'volmix_a': 1.2,
    'volmix_c': 0.5,
    'volmix_volume_z_window_months': 6,

    # Ensemble config (requested)
    'run_ensemble': True,
    'ensemble_experts': [
        'S2L_BASE','S2L_HEAT',
        'S2R_RIDGE_BASE','S2R_RIDGE_HEAT',
        'S2R_LASSO_BASE','S2R_LASSO_HEAT',
        'S2R_NNLS_BASE','S2R_NNLS_HEAT',
        'VOLMIX'
    ],
    'ensemble_standardize': 'rank_z',
    'ensemble_ic_window_months': 12,
    'ensemble_ic_min_periods_months': 6,
    'ensemble_score_lambda_turnover': 0.25,
    'ensemble_score_lambda_ic_std': 0.50,
    'ensemble_softmax_temp': 5.0,
    'ensemble_disagreement_gamma': 0.50,
    'ensemble_min_active_experts': 3,
    'ensemble_methods': ['proportional', 'meanvariance_tc'],
    'ensemble_fallback': 'EQW',
}


def load_data(config):
    tickers = config['tickers']
    start = pd.to_datetime(config.get('start_date')) if config.get('start_date') else None
    end = pd.to_datetime(config.get('end_date')) if config.get('end_date') else None

    raw = pd.read_csv(config['local_csv_path'], header=[0, 1], index_col=0, parse_dates=True)
    px_d = raw['Adj Close'].reindex(columns=tickers).sort_index()

    if 'Volume' in raw.columns.get_level_values(0):
        vol_d = raw['Volume'].reindex(columns=tickers).sort_index()
    else:
        vol_d = pd.DataFrame(index=px_d.index, columns=px_d.columns)

    if start is not None:
        px_d = px_d.loc[px_d.index >= start]
        vol_d = vol_d.loc[vol_d.index >= start]
    if end is not None:
        px_d = px_d.loc[px_d.index <= end]
        vol_d = vol_d.loc[vol_d.index <= end]

    return px_d, vol_d


px_d, vol_d = load_data(CONFIG)
print('data loaded:', px_d.shape, vol_d.shape)


In [ ]:
def _convert_months_to_periods(month_value, scale):
    return max(1, int(round(float(month_value) * scale)))


def resolve_runtime_params(config):
    rebalance = str(config.get('rebalance', 'M')).upper()
    if rebalance == 'M':
        rule = 'ME'
        periods_per_year = 12
        scale = 1.0
        label = 'Monthly (EOM)'
    elif rebalance.startswith('W-'):
        rule = rebalance
        periods_per_year = 52
        scale = 52.0 / 12.0
        label = f'Weekly ({rebalance})'
    else:
        raise ValueError("rebalance must be 'M' or 'W-XXX'.")

    lookbacks_months = [int(x) for x in config['factor_mom_lookbacks']]
    lookbacks_periods = [_convert_months_to_periods(x, scale) for x in lookbacks_months]

    raw_cw = {int(k): float(v) for k, v in config['combine_weights'].items()}
    combined = {}
    for lb_m, lb_p in zip(lookbacks_months, lookbacks_periods):
        combined[lb_p] = combined.get(lb_p, 0.0) + raw_cw.get(lb_m, 0.0)

    if sum(combined.values()) <= 0:
        unique_lb = sorted(set(lookbacks_periods))
        combined = {k: 1.0 / len(unique_lb) for k in unique_lb}
    else:
        s = sum(combined.values())
        combined = {k: v / s for k, v in combined.items()}

    return {
        'rule': rule,
        'periods_per_year': periods_per_year,
        'scale': scale,
        'label': label,
        'pca_window': _convert_months_to_periods(config['pca_window_months'], scale),
        'reg_window': _convert_months_to_periods(config['regression_window_months'], scale),
        'min_obs': _convert_months_to_periods(config['min_obs_per_window'], scale),
        'skip_periods': _convert_months_to_periods(config['factor_mom_skip_months'], scale),
        'lookbacks_periods': sorted(set(lookbacks_periods)),
        'combine_weights_periods': combined,
    }


def resolve_volume_mix_runtime_params(config, runtime):
    return {
        'volume_z_window_periods': _convert_months_to_periods(config.get('volmix_volume_z_window_months', 6), runtime['scale']),
    }


def to_periodic(px_d, vol_d, rule):
    px_p = px_d.resample(rule).last()
    vol_p = vol_d.resample(rule).sum(min_count=1)
    ret_p = px_p.pct_change()
    return px_p, ret_p, vol_p


def build_rolling_pca(ret_p, runtime, n_components=8):
    window = runtime['pca_window']
    min_obs = runtime['min_obs']

    factor_cols = [f'F{i+1}' for i in range(n_components)]
    factor_ret = pd.DataFrame(index=ret_p.index, columns=factor_cols, dtype=float)
    loadings_by_t = {}
    eligibility_mask_by_t = {}

    prev_loadings = None

    for i, t in enumerate(ret_p.index):
        if i < window - 1:
            continue

        win = ret_p.iloc[i - window + 1: i + 1]
        obs = win.notna().sum()
        elig = (obs >= min_obs)
        elig = elig & ret_p.loc[t].notna()
        elig = elig.reindex(ret_p.columns).fillna(False)
        eligibility_mask_by_t[t] = elig

        if int(elig.sum()) < 2:
            continue

        x = win.loc[:, elig.index[elig.values]].copy()
        x = x.apply(lambda col: col.fillna(col.mean()), axis=0).fillna(0.0)

        mu = x.mean(axis=0)
        sd = x.std(axis=0, ddof=0).replace(0.0, 1.0)
        x_std = (x - mu) / sd

        k = min(n_components, x_std.shape[1], x_std.shape[0])
        if k < 1:
            continue

        pca = PCA(n_components=k, random_state=42)
        scores = pca.fit_transform(x_std.values)
        loadings = pd.DataFrame(
            pca.components_.T,
            index=x_std.columns,
            columns=[f'F{i+1}' for i in range(k)],
        )

        if prev_loadings is not None:
            common_assets = loadings.index.intersection(prev_loadings.index)
            common_factors = loadings.columns.intersection(prev_loadings.columns)
            for fc in common_factors:
                if len(common_assets) == 0:
                    continue
                anchor = float((loadings.loc[common_assets, fc] * prev_loadings.loc[common_assets, fc]).sum())
                if anchor < 0:
                    loadings[fc] *= -1.0
                    col_idx = list(loadings.columns).index(fc)
                    scores[:, col_idx] *= -1.0

        prev_loadings = loadings.copy()

        f_t = pd.Series(scores[-1, :], index=loadings.columns)
        factor_ret.loc[t, f_t.index] = f_t.values
        loadings_by_t[t] = loadings

    return factor_ret, loadings_by_t, eligibility_mask_by_t


def compute_factor_mom_component(factor_ret, lookback_period, skip_periods):
    L = int(lookback_period)
    start_lag = 1
    if skip_periods > 0 and L > (skip_periods + 1):
        start_lag = int(skip_periods) + 1

    lagged = [factor_ret.shift(j) for j in range(start_lag, L + 1)]
    if not lagged:
        return factor_ret * np.nan
    return sum(lagged)


def compute_factor_mom(factor_ret, lookbacks_periods, skip_periods, combine_weights_periods):
    lookbacks = sorted(set(int(x) for x in lookbacks_periods))
    raw_w = {int(k): float(v) for k, v in combine_weights_periods.items() if int(k) in lookbacks}

    s = sum(raw_w.values())
    if s <= 0:
        norm_w = {L: 1.0 / len(lookbacks) for L in lookbacks}
    else:
        norm_w = {L: raw_w.get(L, 0.0) / s for L in lookbacks}

    parts = {}
    for L in lookbacks:
        parts[L] = compute_factor_mom_component(factor_ret, L, skip_periods)

    mom = sum(norm_w[L] * parts[L] for L in lookbacks)
    return mom, parts, norm_w


def resolve_excluded_factor_labels(exclude_pcs, factor_columns):
    labels = [c for c in factor_columns if isinstance(c, str) and c.startswith('F') and c[1:].isdigit()]
    valid_nums = {int(c[1:]) for c in labels}

    excluded = set()
    for raw in (exclude_pcs or []):
        try:
            n = int(raw)
            if n in valid_nums:
                excluded.add(f'F{n}')
        except (TypeError, ValueError):
            pass

    return excluded


def apply_factor_exclusion(factor_mom, excluded_labels):
    out = factor_mom.copy()
    cols = [c for c in excluded_labels if c in out.columns]
    if cols:
        out.loc[:, cols] = np.nan
    return out


def _normalize_abs_signal(sig):
    s = sig.fillna(0.0).copy()
    denom = s.abs().sum()
    if denom <= 0:
        return s * 0.0
    return s / denom


def build_factor_signal_components(mom_short_t, mom_long_t, top_m, select_by_abs=True):
    combined = (mom_short_t.fillna(0.0) + mom_long_t.fillna(0.0)).dropna()
    out_short = pd.Series(0.0, index=mom_short_t.index, dtype=float)
    out_long = pd.Series(0.0, index=mom_long_t.index, dtype=float)

    if len(combined) == 0:
        return out_short, out_long

    k = min(int(top_m), len(combined))
    if k <= 0:
        return out_short, out_long

    picked = combined.abs().nlargest(k).index if select_by_abs else combined.nlargest(k).index
    out_short.loc[picked] = mom_short_t.reindex(picked).fillna(0.0).values
    out_long.loc[picked] = mom_long_t.reindex(picked).fillna(0.0).values

    return _normalize_abs_signal(out_short), _normalize_abs_signal(out_long)


def map_signal_to_etf_prediction(exposure_t, signal_t, elig_t):
    all_assets = elig_t.index
    if exposure_t is None or exposure_t.empty:
        return pd.Series(0.0, index=all_assets, dtype=float)

    fac = exposure_t.columns.intersection(signal_t.index)
    if len(fac) == 0:
        return pd.Series(0.0, index=all_assets, dtype=float)

    r_raw = exposure_t[fac].dot(signal_t.reindex(fac).fillna(0.0))
    out = pd.Series(0.0, index=all_assets, dtype=float)
    out.loc[r_raw.index] = r_raw.values
    out.loc[~elig_t] = 0.0
    return out.fillna(0.0)


RUNTIME = resolve_runtime_params(CONFIG)
RUNTIME_VM = resolve_volume_mix_runtime_params(CONFIG, RUNTIME)
px_p, ret_p, vol_p = to_periodic(px_d, vol_d, RUNTIME['rule'])

factor_ret, loadings_by_t, eligibility_mask_by_t = build_rolling_pca(
    ret_p=ret_p,
    runtime=RUNTIME,
    n_components=CONFIG['n_components'],
)

factor_mom_raw, mom_parts, norm_cw = compute_factor_mom(
    factor_ret=factor_ret,
    lookbacks_periods=RUNTIME['lookbacks_periods'],
    skip_periods=RUNTIME['skip_periods'],
    combine_weights_periods=RUNTIME['combine_weights_periods'],
)
excluded = resolve_excluded_factor_labels(CONFIG.get('exclude_pcs', []), factor_mom_raw.columns)
factor_mom = apply_factor_exclusion(factor_mom_raw, excluded)

lookbacks = sorted(RUNTIME['lookbacks_periods'])
short_lb = lookbacks[0]
long_lbs = [x for x in lookbacks if x != short_lb]

mom_short = compute_factor_mom_component(factor_ret, short_lb, RUNTIME['skip_periods'])
if len(long_lbs) == 0:
    mom_long = pd.DataFrame(0.0, index=factor_ret.index, columns=factor_ret.columns)
else:
    cw = RUNTIME['combine_weights_periods']
    wsum = sum(cw.get(lb, 0.0) for lb in long_lbs)
    lw = ({lb: 1.0 / len(long_lbs) for lb in long_lbs} if wsum <= 0 else {lb: cw.get(lb, 0.0) / wsum for lb in long_lbs})
    mom_long = sum(lw[lb] * compute_factor_mom_component(factor_ret, lb, RUNTIME['skip_periods']) for lb in long_lbs)

print(RUNTIME['label'], '| ret_p shape=', ret_p.shape)
print('loadings dates:', len(loadings_by_t), '| factor_mom non-null rows:', int(factor_mom.notna().any(axis=1).sum()))
print('short_lb(period)=', short_lb, '| long_lbs(period)=', long_lbs)


In [ ]:
def normalize_long_only(w):
    w = w.fillna(0.0).clip(lower=0.0)
    s = w.sum()
    return (w / s) if s > 0 else (w * 0.0)


def normalize_ls(w):
    w = w.fillna(0.0)
    gross = w.abs().sum()
    return (w / gross) if gross > 0 else (w * 0.0)


def apply_weight_cap(w, cap=None):
    if cap is None:
        return w
    out = w.copy().clip(lower=0.0, upper=float(cap))
    return normalize_long_only(out)


def _allocate_proportional_positive(score):
    x = score.fillna(0.0)
    out = pd.Series(0.0, index=x.index, dtype=float)
    pos = x[x > 0]
    if len(pos) == 0:
        return out
    out.loc[pos.index] = pos.values
    return normalize_long_only(out)


def build_eqw(elig_t, assets):
    elig = elig_t.reindex(assets).fillna(False).astype(bool)
    out = pd.Series(0.0, index=assets, dtype=float)
    n = int(elig.sum())
    if n > 0:
        out.loc[elig.index[elig.values]] = 1.0 / n
    return out


def compute_turnover(w_t, w_prev):
    return 0.5 * np.abs(w_t.fillna(0.0) - w_prev.fillna(0.0)).sum()


def next_period_index(idx, t):
    if t not in idx:
        return None
    pos = idx.get_loc(t)
    if isinstance(pos, slice):
        pos = pos.stop - 1
    elif isinstance(pos, np.ndarray):
        pos = int(np.where(pos)[0][-1])
    nxt = int(pos) + 1
    if nxt >= len(idx):
        return None
    return idx[nxt]


def run_backtest(weights_by_t, ret_p, cost_bps):
    if weights_by_t.empty:
        idx = pd.DatetimeIndex([], name='period')
        empty = pd.Series(index=idx, dtype=float)
        return empty.copy(), empty.copy(), empty.copy(), empty.copy(), empty.copy()

    weights_by_t = weights_by_t.sort_index().fillna(0.0)
    tickers = list(weights_by_t.columns)

    decision_periods = []
    applied_periods = []
    gross_vals = []
    net_vals = []
    turn_vals = []

    w_prev = pd.Series(0.0, index=tickers)

    for t, w_t in weights_by_t.iterrows():
        apply_t = next_period_index(ret_p.index, t)
        if apply_t is None:
            continue

        r_next = ret_p.loc[apply_t].reindex(tickers).fillna(0.0)
        w_t = w_t.reindex(tickers).fillna(0.0)

        turnover = compute_turnover(w_t, w_prev)
        gross = float(np.dot(w_t.values, r_next.values))
        net = gross - turnover * (cost_bps / 10000.0)

        decision_periods.append(t)
        applied_periods.append(apply_t)
        gross_vals.append(gross)
        net_vals.append(net)
        turn_vals.append(turnover)

        w_prev = w_t

    idx = pd.DatetimeIndex(applied_periods, name='period')
    gross_s = pd.Series(gross_vals, index=idx, name='gross')
    net_s = pd.Series(net_vals, index=idx, name='net')
    turn_s = pd.Series(turn_vals, index=idx, name='turnover')

    equity = (1.0 + net_s).cumprod().rename('equity')
    peak = equity.cummax()
    dd = (equity / peak - 1.0).rename('drawdown')

    decision_idx = pd.DatetimeIndex(decision_periods, name='decision_period')
    for s in (gross_s, net_s, turn_s, equity, dd):
        s.attrs['decision_periods'] = decision_idx

    return gross_s, net_s, turn_s, equity, dd


def calc_metrics(net_ret, turnover, periods_per_year):
    r = net_ret.dropna()
    if len(r) == 0:
        return pd.Series({'CAGR': np.nan, 'Vol': np.nan, 'Sharpe': np.nan, 'MDD': np.nan, 'Turnover': np.nan, 'Hit': np.nan})

    years = len(r) / float(periods_per_year)
    cagr = (1.0 + r).prod() ** (1.0 / years) - 1.0 if years > 0 else np.nan
    vol = r.std(ddof=0) * np.sqrt(periods_per_year)
    sharpe = (r.mean() / r.std(ddof=0) * np.sqrt(periods_per_year)) if r.std(ddof=0) > 0 else np.nan

    equity = (1.0 + r).cumprod()
    mdd = (equity / equity.cummax() - 1.0).min()
    to = turnover.reindex(r.index).mean()
    hit = (r > 0).mean()

    return pd.Series({'CAGR': cagr, 'Vol': vol, 'Sharpe': sharpe, 'MDD': mdd, 'Turnover': to, 'Hit': hit})


def _safe_get_pos(idx, t):
    pos = idx.get_loc(t)
    if isinstance(pos, slice):
        pos = pos.stop - 1
    elif isinstance(pos, np.ndarray):
        pos = int(np.where(pos)[0][-1])
    return int(pos)


def estimate_asset_volatility(ret_p, t, window_periods, assets):
    if t not in ret_p.index:
        return pd.Series(1.0, index=assets, dtype=float)

    pos = _safe_get_pos(ret_p.index, t)
    start = max(0, pos - int(window_periods) + 1)
    win = ret_p.iloc[start: pos + 1].reindex(columns=assets)

    vol = win.std(ddof=0).replace(0.0, np.nan)
    med = vol.median()
    fallback = float(med) if pd.notna(med) and med > 0 else 1.0
    return vol.fillna(fallback).astype(float)


def estimate_covariance_shrunk(ret_p, t, window_periods, assets, shrinkage=0.2):
    n = len(assets)
    if n == 0:
        return pd.DataFrame()

    if t not in ret_p.index:
        return pd.DataFrame(np.eye(n), index=assets, columns=assets)

    pos = _safe_get_pos(ret_p.index, t)
    start = max(0, pos - int(window_periods) + 1)
    win = ret_p.iloc[start: pos + 1].reindex(columns=assets).fillna(0.0)

    if len(win) < 2:
        return pd.DataFrame(np.eye(n) * 1e-4, index=assets, columns=assets)

    sample = win.cov(ddof=0).reindex(index=assets, columns=assets).fillna(0.0)
    diag = np.diag(np.diag(sample.values))

    s = float(np.clip(shrinkage, 0.0, 1.0))
    cov = (1.0 - s) * sample.values + s * diag
    cov = 0.5 * (cov + cov.T)

    min_eig = np.linalg.eigvalsh(cov).min()
    if min_eig < 1e-10:
        cov = cov + np.eye(n) * (1e-10 - min_eig + 1e-12)

    return pd.DataFrame(cov, index=assets, columns=assets)


def solve_meanvariance_tc(
    mu,
    cov,
    w_prev,
    long_only=True,
    weight_cap=None,
    risk_aversion=10.0,
    turnover_penalty=5.0,
    abs_eps=1e-8,
    maxiter=300,
):
    assets = list(mu.index)
    mu = mu.reindex(assets).fillna(0.0).astype(float)
    cov = cov.reindex(index=assets, columns=assets).fillna(0.0).astype(float)
    w_prev = w_prev.reindex(assets).fillna(0.0).astype(float)

    fallback = _allocate_proportional_positive(mu) if long_only else normalize_ls(mu)

    if not long_only:
        fallback.attrs['mv_fallback'] = True
        fallback.attrs['mv_success'] = False
        fallback.attrs['mv_message'] = 'long_only=False is not optimized in current notebook.'
        return fallback

    if len(assets) == 0:
        out = pd.Series(dtype=float)
        out.attrs['mv_fallback'] = True
        out.attrs['mv_success'] = False
        out.attrs['mv_message'] = 'empty asset universe'
        return out

    if (mu > 0).sum() == 0:
        out = pd.Series(0.0, index=assets, dtype=float)
        out.attrs['mv_fallback'] = False
        out.attrs['mv_success'] = True
        out.attrs['mv_message'] = 'no positive expected return'
        return out

    cov_mat = cov.values.astype(float)
    cov_mat = 0.5 * (cov_mat + cov_mat.T)
    min_eig = np.linalg.eigvalsh(cov_mat).min()
    if min_eig < 1e-12:
        cov_mat = cov_mat + np.eye(len(assets)) * (1e-12 - min_eig + 1e-12)

    mu_vec = mu.values
    w_prev_vec = w_prev.values

    w0 = _allocate_proportional_positive(mu).values if np.isclose(w_prev_vec.sum(), 0.0) else normalize_long_only(w_prev).values

    cap = 1.0 if weight_cap is None else float(weight_cap)
    bounds = [(0.0, cap) for _ in assets]
    constraints = [{'type': 'eq', 'fun': lambda w: float(np.sum(w) - 1.0)}]

    lam = float(risk_aversion)
    eta = float(turnover_penalty)
    eps = float(abs_eps)

    def objective(w):
        diff = w - w_prev_vec
        risk = 0.5 * lam * float(w @ cov_mat @ w)
        ret = float(mu_vec @ w)
        tc = eta * float(np.sqrt(diff * diff + eps).sum())
        return risk - ret + tc

    result = minimize(
        objective,
        w0,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'maxiter': int(maxiter), 'ftol': 1e-9, 'disp': False},
    )

    if (not result.success) or (not np.isfinite(result.fun)):
        out = fallback.copy()
        out.attrs['mv_fallback'] = True
        out.attrs['mv_success'] = False
        out.attrs['mv_message'] = str(result.message)
        return out

    w_opt = pd.Series(result.x, index=assets, dtype=float).clip(lower=0.0)
    if weight_cap is not None:
        w_opt = w_opt.clip(upper=float(weight_cap))
    w_opt = normalize_long_only(w_opt)

    w_opt.attrs['mv_fallback'] = False
    w_opt.attrs['mv_success'] = True
    w_opt.attrs['mv_message'] = str(result.message)
    return w_opt


In [ ]:
def build_signed_factor_signal(mom_t, top_m=3, select_by_abs=True):
    out = pd.Series(0.0, index=mom_t.index, dtype=float)
    x = mom_t.dropna().copy()
    if len(x) == 0:
        return out

    k = min(max(0, int(top_m)), len(x))
    if k == 0:
        return out

    picked = x.abs().nlargest(k).index if select_by_abs else x.nlargest(k).index
    out.loc[picked] = x.loc[picked].values

    scale = out.abs().sum()
    if scale > 0:
        out = out / scale
    return out


def predict_etf_returns(exposure_t, factor_signal_t, elig_mask_t):
    return map_signal_to_etf_prediction(exposure_t, factor_signal_t, elig_mask_t)


def _percentile_from_history(history_s, lookback, min_periods):
    x = history_s.dropna().astype(float)
    if lookback is not None and lookback > 0:
        x = x.tail(int(lookback))
    if len(x) < int(min_periods):
        return 0.5

    arr = np.sort(x.values)
    cur = float(arr[-1])
    rank = float(np.searchsorted(arr, cur, side='right')) / float(len(arr))
    return float(np.clip(rank, 0.0, 1.0))


def compute_volume_zscore(vol_p, t, window_periods, assets):
    assets = list(assets)
    if len(assets) == 0:
        return pd.Series(dtype=float)
    if t not in vol_p.index:
        return pd.Series(0.0, index=assets, dtype=float)

    pos = _safe_get_pos(vol_p.index, t)
    start = max(0, pos - int(window_periods) + 1)

    win = np.log1p(vol_p.iloc[start: pos + 1].reindex(columns=assets).astype(float))
    win = win.replace([np.inf, -np.inf], np.nan)

    if len(win) == 0:
        return pd.Series(0.0, index=assets, dtype=float)

    cur = win.iloc[-1]
    mu = win.mean(axis=0)
    sd = win.std(axis=0, ddof=0).replace(0.0, np.nan)

    z = (cur - mu) / sd
    z = z.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return z.astype(float)


def update_heat_state(heat_state, t, disp_t, breadth_t):
    if 'disp_hist' not in heat_state:
        heat_state['disp_hist'] = pd.Series(dtype=float)
    if 'breadth_hist' not in heat_state:
        heat_state['breadth_hist'] = pd.Series(dtype=float)

    heat_state['disp_hist'].loc[t] = float(disp_t)
    heat_state['breadth_hist'].loc[t] = float(breadth_t)


def compute_heat_components_t(t, r_hat_full, vol_p, elig_t, heat_state, config, runtime):
    investable = elig_t.index[elig_t.values]

    r_i = r_hat_full.reindex(investable).fillna(0.0)
    disp_t = float(r_i.std(ddof=0)) if len(r_i) > 0 else 0.0
    breadth_t = float((r_i > 0).mean()) if len(r_i) > 0 else 0.0

    update_heat_state(heat_state, t, disp_t, breadth_t)

    lookback = int(runtime.get('heat_norm_lookback_periods', 24))
    minp = int(runtime.get('heat_norm_min_periods', 12))

    disp_norm = _percentile_from_history(heat_state['disp_hist'], lookback, minp)
    breadth_norm = _percentile_from_history(heat_state['breadth_hist'], lookback, minp)

    if vol_p is None:
        vol_heat_local = pd.Series(0.0, index=investable, dtype=float)
    else:
        z = compute_volume_zscore(
            vol_p=vol_p,
            t=t,
            window_periods=int(runtime.get('heat_volume_z_window_periods', 6)),
            assets=investable,
        )
        if len(z) == 0:
            vol_heat_local = pd.Series(0.0, index=investable, dtype=float)
        else:
            vol_heat_local = z.rank(method='average', pct=True).clip(lower=0.0, upper=1.0)

    vol_heat_global = float(vol_heat_local.mean()) if len(vol_heat_local) > 0 else 0.0

    return {
        'disp_t': disp_t,
        'breadth_t': breadth_t,
        'disp_norm': float(np.clip(disp_norm, 0.0, 1.0)),
        'breadth_norm': float(np.clip(breadth_norm, 0.0, 1.0)),
        'vol_heat_local': vol_heat_local,
        'vol_heat_global': float(np.clip(vol_heat_global, 0.0, 1.0)),
        'investable': investable,
    }


def compose_heat_scores(components_t, config):
    w_disp = float(config.get('heat_dispersion_weight', 1.0/3.0))
    w_br = float(config.get('heat_breadth_weight', 1.0/3.0))
    w_vol = float(config.get('heat_volume_weight', 1.0/3.0))

    w_sum = w_disp + w_br + w_vol
    if w_sum <= 0:
        w_disp = w_br = w_vol = 1.0 / 3.0
        w_sum = 1.0

    w_disp /= w_sum
    w_br /= w_sum
    w_vol /= w_sum

    heat_global = (
        w_disp * float(components_t['disp_norm'])
        + w_br * float(components_t['breadth_norm'])
        + w_vol * float(components_t['vol_heat_global'])
    )
    heat_global = float(np.clip(heat_global, 0.0, 1.0))

    local_blend = float(np.clip(config.get('heat_local_blend', 0.5), 0.0, 1.0))
    vol_local = components_t['vol_heat_local'].copy()
    heat_local = local_blend * heat_global + (1.0 - local_blend) * vol_local
    heat_local = heat_local.clip(lower=0.0, upper=1.0)

    return heat_global, heat_local


def apply_heat_to_signal(r_hat_full, heat_global_t, config):
    alpha = float(config.get('heat_signal_alpha', 0.5))
    floor = float(config.get('heat_signal_floor', 0.2))

    mult = max(float(floor), 1.0 - alpha * float(heat_global_t))
    mult = float(np.clip(mult, 0.0, 1.0))

    return r_hat_full.fillna(0.0) * mult


def apply_heat_to_allocation_score(r_hat_sig, heat_local_s, config):
    beta = float(config.get('heat_alloc_beta', 0.5))
    heat = heat_local_s.reindex(r_hat_sig.index).fillna(0.0).clip(lower=0.0, upper=1.0)
    mult = (1.0 - beta * heat).clip(lower=0.0)
    return r_hat_sig.fillna(0.0) * mult


def estimate_regression_exposure_regularized(
    ret_p,
    factor_ret,
    t,
    window_periods,
    min_obs,
    model='ridge',
    ridge_alpha=5.0,
    lasso_alpha=0.001,
    lasso_max_iter=10000,
    standardize_x=True,
    nnls_fit_intercept=True,
):
    if t not in ret_p.index:
        return pd.DataFrame(), pd.Series(False, index=ret_p.columns)

    pos = _safe_get_pos(ret_p.index, t)
    if pos < int(window_periods) - 1:
        return pd.DataFrame(), pd.Series(False, index=ret_p.columns)

    win_idx = ret_p.index[pos - int(window_periods) + 1: pos + 1]

    valid_factor_cols = [c for c in factor_ret.columns if pd.notna(factor_ret.loc[t, c])]
    if len(valid_factor_cols) == 0:
        return pd.DataFrame(), pd.Series(False, index=ret_p.columns)

    X_all = factor_ret.reindex(win_idx)[valid_factor_cols]
    model_l = str(model).lower()

    beta_rows = {}

    for tk in ret_p.columns:
        if pd.isna(ret_p.loc[t, tk]):
            continue

        y = ret_p.loc[win_idx, tk]
        reg_df = pd.concat([y.rename('y'), X_all], axis=1).dropna()

        need_obs = max(int(min_obs), len(valid_factor_cols) + 1)
        if len(reg_df) < need_obs:
            continue

        yv = reg_df['y'].values.astype(float)
        X = reg_df[valid_factor_cols].values.astype(float)

        if standardize_x:
            mu = X.mean(axis=0)
            sd = X.std(axis=0, ddof=0)
            sd = np.where(sd <= 0, 1.0, sd)
            X_std = (X - mu) / sd
        else:
            mu = np.zeros(X.shape[1])
            sd = np.ones(X.shape[1])
            X_std = X

        if model_l == 'ridge':
            reg = Ridge(alpha=float(ridge_alpha), fit_intercept=True, random_state=42)
            reg.fit(X_std, yv)
            coef_std = reg.coef_.astype(float)
            intercept_std = float(reg.intercept_)

        elif model_l == 'lasso':
            reg = Lasso(alpha=float(lasso_alpha), fit_intercept=True, max_iter=int(lasso_max_iter), random_state=42)
            reg.fit(X_std, yv)
            coef_std = reg.coef_.astype(float)
            intercept_std = float(reg.intercept_)

        elif model_l == 'nnls':
            if nnls_fit_intercept:
                y_mean = float(yv.mean())
                coef_std, _ = nnls(X_std, yv - y_mean)
                intercept_std = y_mean
            else:
                coef_std, _ = nnls(X_std, yv)
                intercept_std = 0.0
            coef_std = coef_std.astype(float)

        else:
            raise ValueError(f'Unknown regularized exposure model: {model}')

        if standardize_x:
            coef = coef_std / sd
            _ = float(intercept_std - np.dot(mu, coef))
        else:
            coef = coef_std
            _ = intercept_std

        beta_rows[tk] = coef

    if not beta_rows:
        return pd.DataFrame(), pd.Series(False, index=ret_p.columns)

    beta_mat = pd.DataFrame.from_dict(beta_rows, orient='index', columns=valid_factor_cols).astype(float)
    elig = pd.Series(False, index=ret_p.columns)
    elig.loc[beta_mat.index] = True

    return beta_mat, elig


def run_expert_predictions_signaware(ret_p, vol_p, loadings_by_t, factor_mom, config, runtime_ex):
    all_tickers = ret_p.columns.tolist()
    requested = [str(x).upper() for x in config.get('ensemble_experts', [])]

    all_candidate_experts = [
        'S2L_BASE', 'S2L_HEAT',
        'S2R_RIDGE_BASE', 'S2R_RIDGE_HEAT',
        'S2R_LASSO_BASE', 'S2R_LASSO_HEAT',
        'S2R_NNLS_BASE', 'S2R_NNLS_HEAT',
    ]
    target_experts = [e for e in all_candidate_experts if e in requested]

    pred_by_expert = {e: {} for e in target_experts}
    elig_by_expert = {e: {} for e in target_experts}

    heat_states = {
        e: {'disp_hist': pd.Series(dtype=float), 'breadth_hist': pd.Series(dtype=float)}
        for e in target_experts if e.endswith('_HEAT')
    }

    heat_global_by_expert = {e: {} for e in heat_states}

    model_sparsity = {'RIDGE': [], 'LASSO': [], 'NNLS': []}

    decision_times = sorted(loadings_by_t.keys())

    for t in decision_times:
        if t not in factor_mom.index:
            continue

        mom_t = factor_mom.loc[t]
        if mom_t.isna().all():
            continue

        factor_signal_t = build_signed_factor_signal(
            mom_t=mom_t,
            top_m=config.get('top_m_factors', 3),
            select_by_abs=True,
        )
        if np.isclose(factor_signal_t.abs().sum(), 0.0):
            continue

        exposure_s2l = loadings_by_t.get(t)
        if exposure_s2l is not None and not exposure_s2l.empty:
            elig_s2l = pd.Series(False, index=all_tickers)
            valid_idx = exposure_s2l.index.intersection(ret_p.columns)
            elig_s2l.loc[valid_idx] = ret_p.loc[t, valid_idx].notna().values

            pred_s2l = predict_etf_returns(exposure_s2l, factor_signal_t, elig_s2l)

            if 'S2L_BASE' in pred_by_expert:
                pred_by_expert['S2L_BASE'][t] = pred_s2l.copy()
                elig_by_expert['S2L_BASE'][t] = elig_s2l.copy()

            if 'S2L_HEAT' in pred_by_expert:
                comps = compute_heat_components_t(t, pred_s2l, vol_p, elig_s2l, heat_states['S2L_HEAT'], config, runtime_ex)
                heat_g, heat_l = compose_heat_scores(comps, config)
                pred_heat = apply_heat_to_signal(pred_s2l, heat_g, config)
                pred_heat = apply_heat_to_allocation_score(pred_heat, heat_l.reindex(pred_heat.index).fillna(heat_g), config)
                pred_by_expert['S2L_HEAT'][t] = pred_heat.copy()
                elig_by_expert['S2L_HEAT'][t] = elig_s2l.copy()
                heat_global_by_expert['S2L_HEAT'][t] = heat_g

        for model in ['ridge', 'lasso', 'nnls']:
            exp_t, elig_t = estimate_regression_exposure_regularized(
                ret_p=ret_p,
                factor_ret=factor_ret,
                t=t,
                window_periods=RUNTIME['reg_window'],
                min_obs=RUNTIME['min_obs'],
                model=model,
                ridge_alpha=config.get('ridge_alpha', 5.0),
                lasso_alpha=config.get('lasso_alpha', 0.001),
                lasso_max_iter=config.get('lasso_max_iter', 10000),
                standardize_x=config.get('exposure_standardize_x', True),
                nnls_fit_intercept=config.get('nnls_fit_intercept', True),
            )
            if exp_t is None or exp_t.empty:
                continue

            model_u = model.upper()
            raw_vals = exp_t.values.astype(float).ravel()
            raw_vals = raw_vals[np.isfinite(raw_vals)]
            if len(raw_vals) > 0:
                model_sparsity[model_u].append(float((np.abs(raw_vals) < 1e-10).mean()))

            pred_base = predict_etf_returns(exp_t, factor_signal_t, elig_t)

            name_base = f'S2R_{model_u}_BASE'
            if name_base in pred_by_expert:
                pred_by_expert[name_base][t] = pred_base.copy()
                elig_by_expert[name_base][t] = elig_t.copy()

            name_heat = f'S2R_{model_u}_HEAT'
            if name_heat in pred_by_expert:
                comps = compute_heat_components_t(t, pred_base, vol_p, elig_t, heat_states[name_heat], config, runtime_ex)
                heat_g, heat_l = compose_heat_scores(comps, config)
                pred_heat = apply_heat_to_signal(pred_base, heat_g, config)
                pred_heat = apply_heat_to_allocation_score(pred_heat, heat_l.reindex(pred_heat.index).fillna(heat_g), config)
                pred_by_expert[name_heat][t] = pred_heat.copy()
                elig_by_expert[name_heat][t] = elig_t.copy()
                heat_global_by_expert[name_heat][t] = heat_g

    # 予測が空のexpertは除外
    pred_by_expert = {k: v for k, v in pred_by_expert.items() if len(v) > 0}
    elig_by_expert = {k: elig_by_expert[k] for k in pred_by_expert.keys()}

    diag = {
        'heat_global_by_expert': heat_global_by_expert,
        'model_sparsity_mean': {
            m: (float(np.mean(v)) if len(v) > 0 else np.nan)
            for m, v in model_sparsity.items()
        },
    }

    return {
        'expert_pred_by_t': pred_by_expert,
        'expert_elig_by_t': elig_by_expert,
        'diag': diag,
    }


In [ ]:
def compute_volume_zscore_etf(vol_p, t, window_periods, assets):
    assets = list(assets)
    if len(assets) == 0:
        return pd.Series(dtype=float)
    if t not in vol_p.index:
        return pd.Series(0.0, index=assets, dtype=float)

    pos = _safe_get_pos(vol_p.index, t)
    start = max(0, pos - int(window_periods) + 1)

    win = np.log1p(vol_p.iloc[start: pos + 1].reindex(columns=assets).astype(float))
    win = win.replace([np.inf, -np.inf], np.nan)

    if len(win) == 0:
        return pd.Series(0.0, index=assets, dtype=float)

    cur = win.iloc[-1]
    mu = win.mean(axis=0)
    sd = win.std(axis=0, ddof=0).replace(0.0, np.nan)

    z = (cur - mu) / sd
    return z.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)


def compute_pi_from_volume_z(z_s, a=1.2, c=0.5):
    x = float(a) * (z_s.fillna(0.0).astype(float) - float(c))
    return 1.0 / (1.0 + np.exp(-x))


def combine_short_long_prediction(rhat_short, rhat_long, pi_s):
    pi = pi_s.reindex(rhat_short.index).fillna(0.5).clip(lower=0.0, upper=1.0)
    kappa = 1.0 - 2.0 * pi
    mix = kappa * rhat_short + rhat_long.reindex(rhat_short.index).fillna(0.0)
    return mix, kappa


def run_expert_predictions_volmix(ret_p, vol_p, loadings_by_t, eligibility_mask_by_t, mom_short, mom_long, config, runtime_vm):
    all_tickers = ret_p.columns.tolist()
    if 'VOLMIX' not in [str(x).upper() for x in config.get('ensemble_experts', [])]:
        return {'expert_pred_by_t': {}, 'expert_elig_by_t': {}, 'diag': {}}

    pred = {}
    elig_store = {}

    diag = {'z': {}, 'pi': {}, 'kappa': {}, 'rhat_short': {}, 'rhat_long': {}, 'rhat_mix': {}}

    for t in sorted(loadings_by_t.keys()):
        if (t not in mom_short.index) or (t not in mom_long.index):
            continue

        short_t = mom_short.loc[t]
        long_t = mom_long.loc[t]
        if short_t.isna().all() and long_t.isna().all():
            continue

        sig_short, sig_long = build_factor_signal_components(
            mom_short_t=short_t,
            mom_long_t=long_t,
            top_m=config.get('top_m_factors', 3),
            select_by_abs=True,
        )

        if np.isclose(sig_short.abs().sum() + sig_long.abs().sum(), 0.0):
            continue

        exposure_t = loadings_by_t[t]
        elig_t = eligibility_mask_by_t[t].reindex(all_tickers).fillna(False).astype(bool)

        rhat_short_t = map_signal_to_etf_prediction(exposure_t, sig_short, elig_t)
        rhat_long_t = map_signal_to_etf_prediction(exposure_t, sig_long, elig_t)

        z_t = compute_volume_zscore_etf(vol_p, t, runtime_vm['volume_z_window_periods'], all_tickers)
        pi_t = compute_pi_from_volume_z(z_t, a=config.get('volmix_a', 1.2), c=config.get('volmix_c', 0.5))
        rhat_mix_t, kappa_t = combine_short_long_prediction(rhat_short_t, rhat_long_t, pi_t)

        pred[t] = rhat_mix_t
        elig_store[t] = elig_t

        diag['z'][t] = z_t
        diag['pi'][t] = pi_t
        diag['kappa'][t] = kappa_t
        diag['rhat_short'][t] = rhat_short_t
        diag['rhat_long'][t] = rhat_long_t
        diag['rhat_mix'][t] = rhat_mix_t

    return {
        'expert_pred_by_t': {'VOLMIX': pred} if len(pred) > 0 else {},
        'expert_elig_by_t': {'VOLMIX': elig_store} if len(elig_store) > 0 else {},
        'diag': diag,
    }


In [ ]:
def resolve_ensemble_runtime_params(config, runtime):
    return {
        'vol_window_periods': _convert_months_to_periods(config.get('vol_window_months', 6), runtime['scale']),
        'cov_window_periods': _convert_months_to_periods(config.get('cov_window_months', 12), runtime['scale']),
        'heat_norm_lookback_periods': _convert_months_to_periods(config.get('heat_norm_lookback_months', 24), runtime['scale']),
        'heat_norm_min_periods': _convert_months_to_periods(config.get('heat_norm_min_periods_months', 12), runtime['scale']),
        'heat_volume_z_window_periods': _convert_months_to_periods(config.get('heat_volume_z_window_months', 6), runtime['scale']),
        'ensemble_ic_window_periods': _convert_months_to_periods(config.get('ensemble_ic_window_months', 12), runtime['scale']),
        'ensemble_ic_min_periods': _convert_months_to_periods(config.get('ensemble_ic_min_periods_months', 6), runtime['scale']),
    }


def standardize_prediction_cs(r_hat_t, elig_t, method='rank_z'):
    x = r_hat_t.reindex(elig_t.index).fillna(0.0)
    x.loc[~elig_t] = np.nan

    if str(method).lower() != 'rank_z':
        out = x.fillna(0.0)
        out.loc[~elig_t] = 0.0
        return out

    valid = x[elig_t].dropna()
    out = pd.Series(0.0, index=x.index, dtype=float)
    if len(valid) == 0:
        return out

    ranks = valid.rank(method='average', pct=True)
    z = (ranks - 0.5) / 0.28867513459
    out.loc[valid.index] = z.values
    out.loc[~elig_t] = 0.0
    return out


def build_expert_panel_at_t(expert_pred_by_t, t, assets, elig_union_t):
    cols = {}
    for e, by_t in expert_pred_by_t.items():
        if t not in by_t:
            continue
        s = by_t[t].reindex(assets).fillna(0.0)
        s.loc[~elig_union_t] = 0.0
        cols[e] = s

    if len(cols) == 0:
        return pd.DataFrame(index=assets)

    return pd.DataFrame(cols, index=assets)


def compute_expert_ic_at_t(pred_panel_t, ret_next_t):
    out = pd.Series(np.nan, index=pred_panel_t.columns, dtype=float)
    y = ret_next_t.reindex(pred_panel_t.index)

    for c in pred_panel_t.columns:
        x = pred_panel_t[c]
        df = pd.concat([x.rename('x'), y.rename('y')], axis=1).dropna()
        if len(df) < 5:
            continue
        if df['x'].std(ddof=0) <= 0 or df['y'].std(ddof=0) <= 0:
            continue
        out.loc[c] = float(df['x'].corr(df['y']))

    return out


def compute_provisional_turnover_at_t(pred_panel_t, prev_weight_by_expert, elig_t):
    to_s = pd.Series(np.nan, index=pred_panel_t.columns, dtype=float)
    new_w = {}

    for c in pred_panel_t.columns:
        mu = pred_panel_t[c].reindex(elig_t.index).fillna(0.0)
        mu.loc[~elig_t] = 0.0
        w = _allocate_proportional_positive(mu)

        prev = prev_weight_by_expert.get(c, pd.Series(0.0, index=elig_t.index))
        prev = prev.reindex(elig_t.index).fillna(0.0)

        to_s.loc[c] = float(compute_turnover(w, prev))
        new_w[c] = w

    return to_s, new_w


def update_expert_history(history, t, ic_t, to_t):
    if 'ic' not in history:
        history['ic'] = pd.DataFrame(dtype=float)
    if 'to' not in history:
        history['to'] = pd.DataFrame(dtype=float)

    ic_clean = pd.to_numeric(ic_t, errors='coerce').astype(float)
    to_clean = pd.to_numeric(to_t, errors='coerce').astype(float)

    ic_row = ic_clean.to_frame().T
    ic_row.index = pd.DatetimeIndex([t])
    to_row = to_clean.to_frame().T
    to_row.index = pd.DatetimeIndex([t])

    history['ic'] = pd.concat([history['ic'], ic_row], axis=0)
    history['to'] = pd.concat([history['to'], to_row], axis=0)

    history['ic'] = history['ic'][~history['ic'].index.duplicated(keep='last')].sort_index()
    history['to'] = history['to'][~history['to'].index.duplicated(keep='last')].sort_index()


def compute_expert_scores(history, t, lookback_periods, min_periods, lambda_to, lambda_ic_std):
    ic_hist = history.get('ic', pd.DataFrame())
    to_hist = history.get('to', pd.DataFrame())

    if ic_hist.empty:
        return pd.Series(dtype=float)

    if len(ic_hist.index) > 0:
        ic_use = ic_hist.loc[ic_hist.index < t]
    else:
        ic_use = ic_hist

    if len(to_hist.index) > 0:
        to_use = to_hist.loc[to_hist.index < t]
    else:
        to_use = to_hist

    experts = sorted(set(ic_use.columns).union(set(to_use.columns)))
    out = pd.Series(np.nan, index=experts, dtype=float)

    for e in experts:
        ic_raw = ic_use.get(e, pd.Series(dtype=float))
        to_raw = to_use.get(e, pd.Series(dtype=float))

        ic_s = pd.to_numeric(ic_raw, errors='coerce').dropna().astype(float).tail(int(lookback_periods))
        to_s = pd.to_numeric(to_raw, errors='coerce').dropna().astype(float).tail(int(lookback_periods))

        if len(ic_s) < int(min_periods):
            continue

        to_mean = float(to_s.mean()) if len(to_s) > 0 else 0.0
        score = float(ic_s.mean()) - float(lambda_to) * to_mean - float(lambda_ic_std) * float(ic_s.std(ddof=0))
        out.loc[e] = score

    return out


def softmax_expert_weights(score_t, temp, min_active=3):
    s = score_t.dropna().astype(float)
    if len(s) < int(min_active):
        return pd.Series(dtype=float)

    tau = float(temp)
    x = tau * s.values
    x = x - np.max(x)
    ex = np.exp(x)
    den = float(ex.sum())
    if den <= 0:
        return pd.Series(dtype=float)

    return pd.Series(ex / den, index=s.index, dtype=float)


def combine_expert_predictions(pred_panel_t, expert_w_t, gamma_disagree):
    active = pred_panel_t.columns.intersection(expert_w_t.index)
    if len(active) == 0:
        z = pd.Series(0.0, index=pred_panel_t.index, dtype=float)
        return z.copy(), z.copy()

    panel = pred_panel_t[active].fillna(0.0)
    w = expert_w_t.reindex(active).fillna(0.0)

    mu_ens = panel.dot(w)
    disagreement = panel.std(axis=1, ddof=0)

    gamma = float(max(0.0, gamma_disagree))
    mu_final = mu_ens / (1.0 + gamma * disagreement.clip(lower=0.0))
    return mu_final, disagreement


def allocate_from_ensemble_mu(mu_t, method, prev_w, ret_p, t, config, runtime_ex, elig_t):
    assets = mu_t.index.tolist()
    method_l = str(method).lower()

    mu = mu_t.reindex(assets).fillna(0.0)
    mu.loc[~elig_t] = 0.0

    if method_l == 'proportional':
        return _allocate_proportional_positive(mu)

    if method_l == 'meanvariance_tc':
        investable = list(elig_t.index[elig_t.values])
        if len(investable) == 0:
            return pd.Series(0.0, index=assets, dtype=float)

        cov_t = estimate_covariance_shrunk(
            ret_p=ret_p,
            t=t,
            window_periods=runtime_ex['cov_window_periods'],
            assets=investable,
            shrinkage=config.get('cov_shrinkage', 0.2),
        )

        mu_sub = mu.reindex(investable).fillna(0.0)
        prev_sub = prev_w.reindex(investable).fillna(0.0)

        w_sub = solve_meanvariance_tc(
            mu=mu_sub,
            cov=cov_t,
            w_prev=prev_sub,
            long_only=config.get('long_only', True),
            weight_cap=config.get('weight_cap'),
            risk_aversion=config.get('mv_risk_aversion', 10.0),
            turnover_penalty=config.get('mv_turnover_penalty', 5.0),
            abs_eps=config.get('mv_abs_eps', 1e-8),
            maxiter=config.get('mv_solver_maxiter', 300),
        )

        w_full = pd.Series(0.0, index=assets, dtype=float)
        w_full.loc[w_sub.index] = w_sub.values
        w_full.loc[~elig_t] = 0.0
        return apply_weight_cap(w_full, cap=config.get('weight_cap'))

    raise ValueError(f'Unknown ensemble allocation method: {method}')


def deduplicate_expert_predictions(expert_pred_by_t, expert_elig_by_t, assets, priority_order):
    keep = []
    dropped = {}
    sig_to_keep = {}

    for e in priority_order:
        if e not in expert_pred_by_t or len(expert_pred_by_t[e]) == 0:
            continue

        vec_parts = []
        for t in sorted(expert_pred_by_t[e].keys()):
            p = expert_pred_by_t[e][t].reindex(assets).fillna(0.0).values.astype(float)
            elig = expert_elig_by_t[e].get(t, pd.Series(False, index=assets)).reindex(assets).fillna(False).astype(int).values
            vec_parts.append(np.round(p, 10))
            vec_parts.append(elig.astype(float))

        if len(vec_parts) == 0:
            continue

        sig_vec = np.concatenate(vec_parts)
        sig = hashlib.sha1(sig_vec.tobytes()).hexdigest()

        if sig not in sig_to_keep:
            sig_to_keep[sig] = e
            keep.append(e)
        else:
            dropped[e] = sig_to_keep[sig]

    pred_f = {e: expert_pred_by_t[e] for e in keep}
    elig_f = {e: expert_elig_by_t[e] for e in keep}
    return pred_f, elig_f, dropped


def run_ensemble_strategy(expert_pred_by_t, expert_elig_by_t, ret_p, config, runtime_ext):
    assets = ret_p.columns.tolist()
    methods = [str(m).lower() for m in config.get('ensemble_methods', ['proportional'])]
    experts = list(expert_pred_by_t.keys())

    decision_times = sorted(set().union(*[set(d.keys()) for d in expert_pred_by_t.values()])) if len(experts) > 0 else []

    weights_by_method = {m: {} for m in methods}
    prev_w_by_method = {m: pd.Series(0.0, index=assets, dtype=float) for m in methods}

    prev_w_by_expert = {e: pd.Series(0.0, index=assets, dtype=float) for e in experts}
    history = {'ic': pd.DataFrame(), 'to': pd.DataFrame()}

    diag = {
        'expert_scores_by_t': {},
        'expert_weights_by_t': {},
        'active_expert_count_by_t': {},
        'mu_ens_by_t': {},
        'mu_final_by_t': {},
        'disagreement_by_t': {},
        'fallback_by_t': {},
        'history_count_before_t': {},
    }

    for t in decision_times:
        avail = [e for e in experts if t in expert_pred_by_t[e]]
        if len(avail) == 0:
            continue

        elig_union = pd.Series(False, index=assets)
        for e in avail:
            elig_e = expert_elig_by_t[e].get(t, pd.Series(False, index=assets)).reindex(assets).fillna(False).astype(bool)
            elig_union = elig_union | elig_e

        if int(elig_union.sum()) == 0:
            continue

        panel_raw = build_expert_panel_at_t({e: expert_pred_by_t[e] for e in avail}, t, assets, elig_union)
        panel_std = pd.DataFrame(index=assets)
        for e in panel_raw.columns:
            panel_std[e] = standardize_prediction_cs(panel_raw[e], elig_union, method=config.get('ensemble_standardize', 'rank_z'))

        diag['history_count_before_t'][t] = int(len(history['ic']))

        score_t = compute_expert_scores(
            history=history,
            t=t,
            lookback_periods=runtime_ext['ensemble_ic_window_periods'],
            min_periods=runtime_ext['ensemble_ic_min_periods'],
            lambda_to=config.get('ensemble_score_lambda_turnover', 0.25),
            lambda_ic_std=config.get('ensemble_score_lambda_ic_std', 0.50),
        ).reindex(panel_std.columns)

        exp_w = softmax_expert_weights(
            score_t=score_t,
            temp=config.get('ensemble_softmax_temp', 5.0),
            min_active=config.get('ensemble_min_active_experts', 3),
        )

        fallback = False
        if len(exp_w) == 0:
            fallback = True
            mu_final = pd.Series(0.0, index=assets, dtype=float)
            mu_ens = mu_final.copy()
            disagreement = pd.Series(0.0, index=assets, dtype=float)
        else:
            mu_final, disagreement = combine_expert_predictions(
                pred_panel_t=panel_std,
                expert_w_t=exp_w,
                gamma_disagree=config.get('ensemble_disagreement_gamma', 0.5),
            )
            mu_ens = panel_std[exp_w.index].fillna(0.0).dot(exp_w)

        for m in methods:
            if fallback and str(config.get('ensemble_fallback', 'EQW')).upper() == 'EQW':
                w_t = build_eqw(elig_union, assets)
            else:
                w_t = allocate_from_ensemble_mu(
                    mu_t=mu_final,
                    method=m,
                    prev_w=prev_w_by_method[m],
                    ret_p=ret_p,
                    t=t,
                    config=config,
                    runtime_ex=runtime_ext,
                    elig_t=elig_union,
                )
                if np.isclose(w_t.sum(), 0.0) and str(config.get('ensemble_fallback', 'EQW')).upper() == 'EQW':
                    w_t = build_eqw(elig_union, assets)

            weights_by_method[m][t] = w_t
            prev_w_by_method[m] = w_t.copy()

        full_score = pd.Series(np.nan, index=panel_std.columns, dtype=float)
        full_score.loc[score_t.dropna().index] = score_t.dropna().values

        full_w = pd.Series(0.0, index=panel_std.columns, dtype=float)
        full_w.loc[exp_w.index] = exp_w.values

        diag['expert_scores_by_t'][t] = full_score
        diag['expert_weights_by_t'][t] = full_w
        diag['active_expert_count_by_t'][t] = int((full_w > 0).sum())
        diag['mu_ens_by_t'][t] = mu_ens
        diag['mu_final_by_t'][t] = mu_final
        diag['disagreement_by_t'][t] = disagreement
        diag['fallback_by_t'][t] = bool(fallback)

        apply_t = next_period_index(ret_p.index, t)
        if apply_t is not None:
            ret_next_t = ret_p.loc[apply_t].reindex(assets).fillna(0.0)
            ic_t = compute_expert_ic_at_t(panel_std, ret_next_t)
            to_t, new_w_expert = compute_provisional_turnover_at_t(panel_std, prev_w_by_expert, elig_union)
            update_expert_history(history, t, ic_t, to_t)
            for e, w in new_w_expert.items():
                prev_w_by_expert[e] = w

    strategy_names = {
        'proportional': 'ENS_ICSOFT_PROPORTIONAL',
        'meanvariance_tc': 'ENS_ICSOFT_MEANVARIANCE_TC',
    }

    results = {}
    weights_out = {}
    for m in methods:
        w_df = pd.DataFrame.from_dict(weights_by_method[m], orient='index').reindex(columns=assets).fillna(0.0).sort_index()
        gross, net, turn, eq, dd = run_backtest(w_df, ret_p, config.get('cost_bps', 10))

        name = strategy_names.get(m, f'ENS_{str(m).upper()}')
        weights_out[name] = w_df
        results[name] = {
            'gross': gross,
            'net': net,
            'turnover': turn,
            'equity': eq,
            'drawdown': dd,
        }

    return {
        'weights_by_strategy': weights_out,
        'results_by_strategy': results,
        'diag': diag,
        'history': history,
    }


def run_ensemble_diagnostics(ensemble_res, expert_pred_by_t):
    hist_ic = ensemble_res['history'].get('ic', pd.DataFrame())
    hist_to = ensemble_res['history'].get('to', pd.DataFrame())

    rows = []
    for e in sorted(expert_pred_by_t.keys()):
        ic_raw = hist_ic.get(e, pd.Series(dtype=float)) if not hist_ic.empty else pd.Series(dtype=float)
        to_raw = hist_to.get(e, pd.Series(dtype=float)) if not hist_to.empty else pd.Series(dtype=float)

        ic_s = pd.to_numeric(ic_raw, errors='coerce').dropna().astype(float)
        to_s = pd.to_numeric(to_raw, errors='coerce').dropna().astype(float)

        rows.append({
            'expert': e,
            'decision_count': int(len(expert_pred_by_t[e])),
            'ic_count': int(len(ic_s)),
            'ic_mean': float(ic_s.mean()) if len(ic_s) > 0 else np.nan,
            'ic_std': float(ic_s.std(ddof=0)) if len(ic_s) > 0 else np.nan,
            'turnover_mean': float(to_s.mean()) if len(to_s) > 0 else np.nan,
        })

    expert_stats = pd.DataFrame(rows).set_index('expert').sort_index() if rows else pd.DataFrame()

    active_counts = pd.Series(ensemble_res['diag'].get('active_expert_count_by_t', {}), dtype=float)
    fallback_flags = pd.Series(ensemble_res['diag'].get('fallback_by_t', {}), dtype=bool)

    summary = {
        'active_expert_count_mean': float(active_counts.mean()) if len(active_counts) > 0 else np.nan,
        'active_expert_count_min': float(active_counts.min()) if len(active_counts) > 0 else np.nan,
        'min_active_requirement': float(CONFIG.get('ensemble_min_active_experts', 3)),
        'fallback_ratio': float(fallback_flags.mean()) if len(fallback_flags) > 0 else np.nan,
        'active_ge_min_ratio': float((active_counts >= CONFIG.get('ensemble_min_active_experts', 3)).mean()) if len(active_counts) > 0 else np.nan,
    }

    return {'expert_stats': expert_stats, 'summary': summary}


RUNTIME_ENS = resolve_ensemble_runtime_params(CONFIG, RUNTIME)
print('runtime_ext:', RUNTIME_ENS)


In [ ]:
# 1) expert予測の生成
sa_out = run_expert_predictions_signaware(
    ret_p=ret_p,
    vol_p=vol_p,
    loadings_by_t=loadings_by_t,
    factor_mom=factor_mom,
    config=CONFIG,
    runtime_ex=RUNTIME_ENS,
)
vm_out = run_expert_predictions_volmix(
    ret_p=ret_p,
    vol_p=vol_p,
    loadings_by_t=loadings_by_t,
    eligibility_mask_by_t=eligibility_mask_by_t,
    mom_short=mom_short,
    mom_long=mom_long,
    config=CONFIG,
    runtime_vm=RUNTIME_VM,
)

expert_pred_by_t = {}
expert_elig_by_t = {}

for src in [sa_out, vm_out]:
    for e, d in src.get('expert_pred_by_t', {}).items():
        expert_pred_by_t[e] = d
    for e, d in src.get('expert_elig_by_t', {}).items():
        expert_elig_by_t[e] = d

assets = ret_p.columns.tolist()
priority = [x for x in CONFIG.get('ensemble_experts', []) if x in expert_pred_by_t]
expert_pred_by_t, expert_elig_by_t, dropped_map = deduplicate_expert_predictions(
    expert_pred_by_t=expert_pred_by_t,
    expert_elig_by_t=expert_elig_by_t,
    assets=assets,
    priority_order=priority,
)

# 2) EQW baseline
weights_eqw_dict = {}
for t, elig_t in sorted(eligibility_mask_by_t.items()):
    weights_eqw_dict[t] = build_eqw(elig_t, assets)
weights_eqw = pd.DataFrame.from_dict(weights_eqw_dict, orient='index').reindex(columns=assets).fillna(0.0).sort_index()
gross_eqw, net_eqw, turn_eqw, eq_eqw, dd_eqw = run_backtest(weights_eqw, ret_p, CONFIG['cost_bps'])

# 3) expert単体（proportional）
expert_single_results = {}
expert_single_weights = {}
for e, pred_dict in expert_pred_by_t.items():
    rows = {}
    for t, mu_t in pred_dict.items():
        elig_t = expert_elig_by_t[e].get(t, pd.Series(False, index=assets)).reindex(assets).fillna(False).astype(bool)
        w_t = _allocate_proportional_positive(mu_t.reindex(assets).fillna(0.0).where(elig_t, 0.0))
        if np.isclose(w_t.sum(), 0.0):
            continue
        rows[t] = w_t

    w_df = pd.DataFrame.from_dict(rows, orient='index').reindex(columns=assets).fillna(0.0).sort_index()
    gross, net, turn, eq, dd = run_backtest(w_df, ret_p, CONFIG['cost_bps'])

    expert_single_weights[e] = w_df
    expert_single_results[e] = {
        'gross': gross,
        'net': net,
        'turnover': turn,
        'equity': eq,
        'drawdown': dd,
    }

print('Experts generated:', sorted(expert_pred_by_t.keys()))
print('Dropped duplicated experts:', dropped_map)
print('EQW periods:', len(net_eqw), '| Expert count:', len(expert_single_results))
print('S2R sparsity mean:', sa_out.get('diag', {}).get('model_sparsity_mean', {}))


In [ ]:
# 4) アンサンブル実行
ensemble_res = run_ensemble_strategy(
    expert_pred_by_t=expert_pred_by_t,
    expert_elig_by_t=expert_elig_by_t,
    ret_p=ret_p,
    config=CONFIG,
    runtime_ext=RUNTIME_ENS,
)

ens_results = ensemble_res['results_by_strategy']
ens_weights = ensemble_res['weights_by_strategy']

# 5) 比較テーブル
strategy_ret = {'EQW': net_eqw}
strategy_to = {'EQW': turn_eqw}
strategy_eq = {'EQW': eq_eqw}
strategy_dd = {'EQW': dd_eqw}

for e, res in expert_single_results.items():
    strategy_ret[e] = res['net']
    strategy_to[e] = res['turnover']
    strategy_eq[e] = res['equity']
    strategy_dd[e] = res['drawdown']

for name, res in ens_results.items():
    strategy_ret[name] = res['net']
    strategy_to[name] = res['turnover']
    strategy_eq[name] = res['equity']
    strategy_dd[name] = res['drawdown']

core_names = ['EQW', 'ENS_ICSOFT_PROPORTIONAL', 'ENS_ICSOFT_MEANVARIANCE_TC']
core_names = [n for n in core_names if n in strategy_ret]
common_core = None
for n in core_names:
    idx = strategy_ret[n].index
    common_core = idx if common_core is None else common_core.intersection(idx)

if common_core is None or len(common_core) == 0:
    raise ValueError('No common period among EQW and ensemble strategies.')

metrics_rows = []
for name, r in strategy_ret.items():
    idx = common_core.intersection(r.index)
    if len(idx) == 0:
        continue

    m = calc_metrics(
        net_ret=r.reindex(idx),
        turnover=strategy_to[name].reindex(idx),
        periods_per_year=RUNTIME['periods_per_year'],
    )
    row = {'strategy': name, 'obs': int(len(idx))}
    row.update(m.to_dict())
    metrics_rows.append(row)

metrics_matrix = pd.DataFrame(metrics_rows).set_index('strategy').sort_values('Sharpe', ascending=False)

baseline = metrics_matrix.loc['EQW']
delta_vs_eqw = metrics_matrix[['CAGR', 'Vol', 'Sharpe', 'MDD', 'Turnover', 'Hit']].subtract(baseline, axis=1)
delta_vs_eqw = delta_vs_eqw.rename(columns=lambda c: f'{c}_delta')

rank_table = metrics_matrix[['Sharpe', 'CAGR', 'MDD', 'Turnover']].sort_values('Sharpe', ascending=False)

print(f'Common core window: {common_core.min().date()} -> {common_core.max().date()} ({len(common_core)} periods)')
print('Metrics matrix:')
display(metrics_matrix)
print('Delta vs EQW:')
display(delta_vs_eqw)
print('Rank table (Sharpe):')
display(rank_table)

# 可視化（Sharpe上位8）
top_n = min(8, len(rank_table))
top_names = rank_table.head(top_n).index.tolist()

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
for n in top_names:
    eq = (1.0 + strategy_ret[n].reindex(common_core).fillna(0.0)).cumprod()
    dd = eq / eq.cummax() - 1.0
    axes[0].plot(eq.index, eq.values, linewidth=1.8, label=n)
    axes[1].plot(dd.index, dd.values, linewidth=1.6, label=n)

axes[0].set_title('Top Sharpe Strategies | Equity')
axes[1].set_title('Top Sharpe Strategies | Drawdown')
axes[0].legend(ncol=2, fontsize=9)
axes[1].legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# 6) 妥当性テスト

# (1) 時点整合: t決定 -> t+1適用
for name, res in ens_results.items():
    decision_idx = res['net'].attrs.get('decision_periods', pd.DatetimeIndex([]))
    for t in decision_idx:
        nxt = next_period_index(ret_p.index, t)
        assert nxt is not None and nxt > t, f'timing mismatch in {name}'

# (2) 値域・制約: expert重み
for t, w in ensemble_res['diag']['expert_weights_by_t'].items():
    if len(w) == 0:
        continue
    assert (w.values >= -1e-12).all(), 'expert weight negative'
    s = float(w.sum())
    if s > 0:
        assert np.isclose(s, 1.0, atol=1e-8), 'expert weight sum != 1'

# (3) ポートフォリオ制約
for name, w_df in ens_weights.items():
    if w_df.empty:
        continue
    assert (w_df.values >= -1e-12).all(), f'negative weight in {name}'
    row_sum = w_df.sum(axis=1)
    active = row_sum > 1e-12
    if active.any():
        assert np.allclose(row_sum[active].values, 1.0, atol=1e-8), f'active sum!=1 in {name}'
    if (~active).any():
        assert np.allclose(row_sum[~active].values, 0.0, atol=1e-8), f'inactive sum!=0 in {name}'

# (4) 履歴窓健全性の補助チェック
hist_ic = ensemble_res['history']['ic']
hist_to = ensemble_res['history']['to']
if not hist_ic.empty:
    assert (hist_ic.index == hist_to.index).all(), 'IC/TO history index mismatch'

# (5) 数値健全性
assert len(common_core) > 0, 'common_core is empty'
assert np.isfinite(metrics_matrix[['CAGR','Vol','Sharpe','MDD','Turnover','Hit']].values).all(), 'metrics has NaN/inf'

# (6) softmax単調性
s1 = pd.Series({'A': 0.2, 'B': 0.0})
s2 = pd.Series({'A': 0.4, 'B': 0.0})
w1 = softmax_expert_weights(s1, temp=5.0, min_active=2)
w2 = softmax_expert_weights(s2, temp=5.0, min_active=2)
if len(w1) == 2 and len(w2) == 2:
    assert w2['A'] >= w1['A'] - 1e-12, 'softmax monotonicity failed'

# (7) disagreement増加で|mu_final|抑制（合成関数の構造検証）
panel_test = pd.DataFrame({'E1': [0.2, 0.2], 'E2': [0.2, -0.2]}, index=['X', 'Y'])
exp_w_test = pd.Series({'E1': 0.5, 'E2': 0.5})
mu_final_test, dis_test = combine_expert_predictions(panel_test, exp_w_test, gamma_disagree=0.5)
assert abs(mu_final_test['Y']) <= abs((panel_test.loc['Y'] @ exp_w_test)) + 1e-12, 'disagreement shrink failed'

# (8) expert有効性診断
diag_summary = run_ensemble_diagnostics(ensemble_res, expert_pred_by_t)
expert_stats = diag_summary['expert_stats']
summary = diag_summary['summary']

display(expert_stats)
print('diagnostic summary:', summary)

print('All validation checks passed.')


## まとめ

- `EQW`・expert単体・`ENS_ICSOFT_PROPORTIONAL`・`ENS_ICSOFT_MEANVARIANCE_TC` を同一ノートで比較可能にした。
- 予測統合は `Rolling IC Softmax`、不一致縮小は `1/(1+\gamma d)` で反映。
- 以後の拡張候補は、`ensemble_softmax_temp` と `lambda` 群の感度分析、および expert集合の定期的見直し。
